<a href="https://colab.research.google.com/github/bigbirdjones/laguardia_data_analytics/blob/main/NYC_Cooling_Tower_System_Inspection_Results_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: LOAD THE DATA

## Main Dataset

I'll load my main dataset via API in 3 segments, then concatinate to create a full DataFrame.

My Dataset are results from inspections conducted by the Department of Health & Mental Hygiene (DOHMH) on cooling towers for large HVAC installations around the city.

These violations are covered by Chapter 8 of Title 24 of the Rules of The City of New York. (24 RCNY)

In [4]:
import pandas as pd

In [5]:
URL = "https://data.cityofnewyork.us/resource/f9wb-g8mb.csv?$limit=50000"

df1 = pd.read_csv(URL)

/tmp/ipykernel_34368/2484009375.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(URL)


In [6]:
URL = "https://data.cityofnewyork.us/resource/f9wb-g8mb.csv?$limit=50000&$offset=50000"

df2 = pd.read_csv(URL)

In [9]:
URL = "https://data.cityofnewyork.us/resource/f9wb-g8mb.csv?$limit=22000&$offset=100000"

df3 = pd.read_csv(URL)

In [10]:
full_df = pd.concat([df1, df2, df3])
full_df

,bin,system_id,address,borough,zip_code,status,active_equip,inspection_date,violation_code,law_section,...,citation_text,summons_number,inspection_type,bbl,latitude,longitude,community_board,council_district,census_tract,nta_code
0,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8U,24 RCNY §8-05(f)(5),...,NaN,196816281.0,Non-Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
1,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8C,24 RCNY §8-04(a),...,Respondent is in violation of 24 RCNY 8-04(a)(...,881249959.0,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
2,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8D,24 RCNY §8-04(b),...,Compliance inspections must be completed by a ...,881249968.0,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
3,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8E,24 RCNY §8-04(c),...,"At the time of inspection, operational records...",NaN,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
4,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8L,24 RCNY §8-05(c)(3),...,"At the time of inspection, upon Inspector's re...",881249977.0,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,4000465,2000010231,46-02 21ST ST,Queens,11101,Active,1,2025-09-25T00:00:00.000,AH8E,24 RCNY §8-04(c),...,Respondent failed to demonstrate proof that al...,NaN,Cycle,4000550001,40.745716,-73.948620,0.0,NaN,702.0,QN0201
21996,4000465,2000010231,46-02 21ST ST,Queens,11101,Active,1,2025-09-25T00:00:00.000,AH8C,24 RCNY §8-04(a),...,Respondent failed to provide records of all re...,NaN,Cycle,4000550001,40.745716,-73.948620,0.0,NaN,702.0,QN0201
21997,4000465,2000010231,46-02 21ST ST,Queens,11101,Active,1,2025-09-25T00:00:00.000,AH8A,24 RCNY §8-03,...,Respondent's required Maintenance Program and ...,NaN,Cycle,4000550001,40.745716,-73.948620,0.0,NaN,702.0,QN0201
21998,1088221,2000001962,2 AVENUE OF THE AMERICAS,Manhattan,10013,Active,2,2025-09-25T00:00:00.000,NaN,NaN,...,NaN,NaN,Cycle,1001910001,40.719358,-74.004845,NaN,NaN,3300.0,MN0102


## Auxillary Dataset

Next i'll load my auxillary dataset, which after cleaning I will merge with the main dataset.

These records are derived from the Office of Administrative Trials & Hearings.

They have +20 Million records spanning over a decade, so I queried the records on NYC Open Data to get only the ones from the issuing agency i'm looking for, which is 'COOLING TOWERS - DOHMH', and downloaded the results in a csv.




In [11]:
URL2 = "/content/drive/MyDrive/NYC Cooling Tower EDA Capstone/OATH_Hearings_Division_Case_Status_20260409.csv"
oath_df = pd.read_csv(URL2)

/tmp/ipykernel_34368/2915690999.py:2: DtypeWarning: Columns (31,42,43,44,45,46,47,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  oath_df = pd.read_csv(URL2)


I'm dropping these columns immediately because they are obviously empty from even a cursory glance at the spreadsheet.

They are for charges 2 thru 10, but in the cases i'm looking at, the DOHMH seems to have only issued a single charge per ticketed violation, so it's one charge per row.

OATH handles administrative hearings for multiple city agencies, so it makes sense to leave space, but goodbye empty columns!

In [12]:
oath_df = oath_df.drop(oath_df.columns[[42,43,44,45,46,47,48,49,50,
                                        51,52,53,54,55,56,57,58,59,60,
                                        61,62,63,64,65,66,67,68,69,70,
                                        71,72,73,74,75,76,77]], axis=1)

(I know this is technically cleaning, but it's my notebook and I'll decide where the title breaks go... speaking of which)

In [ ]:
# (I know this is technically cleaning, but it's my notebook and I'll decide where the title breaks go... speaking of which)

#Step 2: CLEAN THE DATA

## Dropping NaN's

First, anyone who wasn't issued a violation I'm not interested in, so NaN's in the **'violation_code'** column are my first target.

In [13]:
full_df["violation_code"].isnull().sum()

np.int64(33795)

In [14]:
full_df.dropna(subset="violation_code", inplace= True)

In [15]:
print(122000-33795)

88205


Next, the **'summons_number'** is how I'm linking my main data set with the **'Ticket Number'** in the auxillary dataset, so NaN's there need to be dealt with as well.

In [16]:
full_df['summons_number'].isnull().sum()

np.int64(33761)

In [17]:
full_df.dropna(subset='summons_number', inplace= True)

In [18]:
print(88205-33761)

54444


Then a quick datatype recast from float64 to Int64 so the key columns will match.

In [19]:
full_df['summons_number'] = full_df['summons_number'].astype(int)

In [20]:
full_df

,bin,system_id,address,borough,zip_code,status,active_equip,inspection_date,violation_code,law_section,...,citation_text,summons_number,inspection_type,bbl,latitude,longitude,community_board,council_district,census_tract,nta_code
0,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8U,24 RCNY §8-05(f)(5),...,NaN,196816281,Non-Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
1,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8C,24 RCNY §8-04(a),...,Respondent is in violation of 24 RCNY 8-04(a)(...,881249959,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
2,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8D,24 RCNY §8-04(b),...,Compliance inspections must be completed by a ...,881249968,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
4,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8L,24 RCNY §8-05(c)(3),...,"At the time of inspection, upon Inspector's re...",881249977,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
7,1015834,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,0,2021-05-21T00:00:00.000,AH8S,24 RCNY §8-05(f)(3),...,"At the time of inspection, records provided by...",881249986,Cycle,1008330078,40.748260,-73.988491,105.0,3.0,7600.0,MN0501
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21925,1081661,2000000725,281 1ST AVE,Manhattan,10003,Active,1,2025-10-06T00:00:00.000,AH8F,24 RCNY §8-04(d),...,The cooling tower system must be cleaned at le...,881327729,Cycle,1009220046,40.733328,-73.982345,NaN,NaN,4800.0,MN0602
21927,1081661,2000000725,281 1ST AVE,Manhattan,10003,Active,1,2025-10-06T00:00:00.000,AH8L,24 RCNY §8-05(c)(3),...,"At the time of inspection, the onsite represen...",881327738,Cycle,1009220046,40.733328,-73.982345,NaN,NaN,4800.0,MN0602
21929,1081661,2000000725,281 1ST AVE,Manhattan,10003,Active,1,2025-10-06T00:00:00.000,AH8Q,24 RCNY §8-05(f)(1),...,"At the time of inspection, operational records...",881327747,Cycle,1009220046,40.733328,-73.982345,NaN,NaN,4800.0,MN0602
21934,1083796,2000014331,636 11TH AVE,Manhattan,10036,Active,1,2025-10-03T00:00:00.000,AH8U,24 RCNY §8-05(f)(5),...,"At the time of inspection, operational records...",881327399,Cycle,1010750001,40.763676,-73.996220,104.0,3.0,12902.0,MN0402


In [21]:
oath_df

,Ticket Number,Violation Date,Violation Time,Issuing Agency,Respondent First Name,Respondent Last Name,Balance Due,Violation Location (Borough),Violation Location (Block No.),Violation Location (Lot No.),...,Respondent Address or Facility Number(For FDNY and DOB Tickets),Penalty Imposed,Paid Amount,Additional Penalties or Late Fees,Compliance Status,Violation Description,Charge #1: Code,Charge #1: Code Section,Charge #1: Code Description,Charge #1: Infraction Amount
0,193687506,NaN,10:30:00,COOLING TOWERS - DOHMH,NaN,EIB KEITREL LLC,$0.00,BROOKLYN,5320.0,24.0,...,NaN,$0.00,$0.00,$0.00,All Terms Met,NaN,AH7A,24RCNY31-02 A,FAIL TO SUBMIT REPORT OF PREVIOUS YEAR'S INSPE...,$250.00
1,193688193,NaN,10:30:00,COOLING TOWERS - DOHMH,NaN,919,$0.00,MANHATTAN,300.0,10.0,...,NaN,$0.00,$0.00,$0.00,All Terms Met,NaN,AH7A,24RCNY31-02 A,FAIL TO SUBMIT REPORT OF PREVIOUS YEAR'S INSPE...,$250.00
2,193662087,NaN,09:52:00,COOLING TOWERS - DOHMH,NaN,ST BARTHOLOMEWS CHURCH,$0.00,MANHATTAN,1305.0,1.0,...,NaN,$0.00,$0.00,$0.00,All Terms Met,NaN,AH8C,24RCNY8-04 A,"ROUTINE MONIT NOT CONDUCT,DOC'D AT LEAST ONCE ...",$500.00
3,193677267,NaN,10:30:00,COOLING TOWERS - DOHMH,NaN,55 LIBERTY OWNERS CORP,$0.00,MANHATTAN,64.0,8.0,...,NaN,$0.00,$0.00,$0.00,All Terms Met,NaN,AH7A,24RCNY31-02 A,FAIL TO SUBMIT REPORT OF PREVIOUS YEAR'S INSPE...,$250.00
4,193671812,NaN,09:54:00,COOLING TOWERS - DOHMH,NaN,ANDREW J TUNICK AS TRUSTEE,$0.00,MANHATTAN,867.0,1.0,...,NaN,$0.00,$0.00,$0.00,All Terms Met,NaN,AH8B,24RCNY8-03,MAINTENANCE PROGRAM AND PLAN INCOMPLETE OR NOT...,$500.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58333,880698390,09/21/2015,13:34:00,COOLING TOWERS - DOHMH,NaN,STERLING LANDLORD CORP,$0.00,MANHATTAN,1022.0,35.0,...,NaN,"$2,000.00","$2,000.00",$0.00,All Terms Met,NaN,AH3K,NYCHC 3.05,"FAILING TO COMPLY WITH DEPT, BOARD OF HEALTH, ...","$1,000.00"
58334,880698345,09/21/2015,13:34:00,COOLING TOWERS - DOHMH,NaN,261 FIFTH AVENUE TIC,"$2,000.00",MANHATTAN,858.0,78.0,...,NaN,"$2,000.00",$0.00,$0.00,Penalty Due,NaN,AH3K,NYCHC 3.05,"FAILING TO COMPLY WITH DEPT, BOARD OF HEALTH, ...","$1,000.00"
58335,880698428,09/21/2015,13:34:00,COOLING TOWERS - DOHMH,NaN,WOODWARD AFFLILIATES,"$2,000.00",MANHATTAN,1026.0,41.0,...,NaN,"$2,000.00",$0.00,$0.00,Penalty Due,NaN,AH3K,NYCHC 3.05,"FAILING TO COMPLY WITH DEPT, BOARD OF HEALTH, ...","$1,000.00"
58336,880698868,09/21/2015,13:34:00,COOLING TOWERS - DOHMH,NaN,COLLEGE POINT HOLDINGS LLC,"$2,000.00",QUEENS,4172.0,7.0,...,NaN,"$2,000.00",$0.00,$0.00,Penalty Due,NaN,AH3K,NYCHC 3.05,"FAILING TO COMPLY WITH DEPT, BOARD OF HEALTH, ...","$1,000.00"


## Merge DataFrames

In [22]:
combo_df = pd.merge(full_df, oath_df, left_on='summons_number', right_on='Ticket Number')

In [23]:
combo_df.columns

Index(['bin', 'system_id', 'address', 'borough', 'zip_code', 'status',
       'active_equip', 'inspection_date', 'violation_code', 'law_section',
       'violation_text', 'violation_type', 'citation_text', 'summons_number',
       'inspection_type', 'bbl', 'latitude', 'longitude', 'community_board',
       'council_district', 'census_tract', 'nta_code', 'Ticket Number',
       'Violation Date', 'Violation Time', 'Issuing Agency',
       'Respondent First Name', 'Respondent Last Name', 'Balance Due',
       'Violation Location (Borough)', 'Violation Location (Block No.)',
       'Violation Location (Lot No.)', 'Violation Location (House #)',
       'Violation Location (Street Name)', 'Violation Location (Floor)',
       'Violation Location (City)', 'Violation Location (Zip Code)',
       'Violation Location (State Name)', 'Respondent Address (Borough)',
       'Respondent Address (House #)', 'Respondent Address (Street Name)',
       'Respondent Address (City)', 'Respondent Address (Zip

There's a lot of redundant info here, so lets make something a little cleaner.

In [24]:
violations_df = combo_df[['Ticket Number', 'system_id', 'address', 'borough', 'zip_code', 'status', 'inspection_date','inspection_type', 'bbl', 'latitude', 'longitude', 'Violation Date', 'violation_code', 'Charge #1: Code Description', 'Respondent Last Name', 'Penalty Imposed', 'Paid Amount', 'Additional Penalties or Late Fees', 'Compliance Status']]

In [26]:
violations_df.set_index('inspection_date')

,Ticket Number,system_id,address,borough,zip_code,status,inspection_type,bbl,latitude,longitude,Violation Date,violation_code,Charge #1: Code Description,Respondent Last Name,Penalty Imposed,Paid Amount,Additional Penalties or Late Fees,Compliance Status
inspection_date,,,,,,,,,,,,,,,,,,
2021-05-21T00:00:00.000,196816281,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,Non-Cycle,1008330078,40.748260,-73.988491,05/21/2021,AH8U,REQUIRED CORRECTIVE ACTION NOT TAKEN BASED ON ...,THE JANE H GOLDMAN 2008Y-11,"$2,000.00","$2,204.00",$0.00,All Terms Met
2021-05-21T00:00:00.000,881249959,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,Cycle,1008330078,40.748260,-73.988491,05/21/2021,AH8C,"ROUTINE MONIT NOT CONDUCT,DOC D AT LEAST ONCE ...",C O SOLIL MANAGEMENT LLC,$500.00,$500.00,$0.00,All Terms Met
2021-05-21T00:00:00.000,881249968,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,Cycle,1008330078,40.748260,-73.988491,05/21/2021,AH8D,"COMPL INSPEC NOT CONDUCT,DOC D ONCE EVERY 90 D...",C O SOLIL MANAGEMENT LLC,$500.00,$500.00,$0.00,All Terms Met
2021-05-21T00:00:00.000,881249977,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,Cycle,1008330078,40.748260,-73.988491,05/21/2021,AH8L,NO RECORDS OF ALL CHEMICALS AND BIOCIDES ADDED...,C O SOLIL MANAGEMENT LLC,$0.00,$0.00,$0.00,All Terms Met
2021-05-21T00:00:00.000,881249986,2000000122,1271 BROADWAY,Manhattan,10001,Inactive,Cycle,1008330078,40.748260,-73.988491,05/21/2021,AH8S,LEGIONELLA SAMP NOT COLLECT ANALYZE RESULTS NO...,C O SOLIL MANAGEMENT LLC,"$1,000.00","$1,000.00",$0.00,All Terms Met
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-06T00:00:00.000,881327729,2000000725,281 1ST AVE,Manhattan,10003,Active,Cycle,1009220046,40.733328,-73.982345,10/06/2025,AH8F,TWICE YEARLY OR OTHER REQUIRED CLEANING NOT CO...,BETH ISRAEL HOSPITAL,$500.00,$0.00,$0.00,Penalty Due
2025-10-06T00:00:00.000,881327738,2000000725,281 1ST AVE,Manhattan,10003,Active,Cycle,1009220046,40.733328,-73.982345,10/06/2025,AH8L,NO RECORDS OF ALL CHEMICALS AND BIOCIDES ADDED...,BETH ISRAEL HOSPITAL,$500.00,$0.00,$0.00,Penalty Due
2025-10-06T00:00:00.000,881327747,2000000725,281 1ST AVE,Manhattan,10003,Active,Cycle,1009220046,40.733328,-73.982345,10/06/2025,AH8Q,MINIMUM DAILY WATER QUALITY MEASUREMENTS NOT T...,BETH ISRAEL HOSPITAL,$500.00,$0.00,$0.00,Penalty Due


# Step 3: ANALYZE THE DATA

In [27]:
import seaborn as sns